# Pharmaceutical Document Q&A System with Intelligent RAG Pipeline

This notebook demonstrates how to build a production-ready document Q&A system that combines:

- Intelligent multi-document detection and classification for pharmaceutical documents
- Query routing for improved retrieval accuracy
- Rich metadata preservation
- User-friendly Gradio interface

## 📚 Setup and Installation
First, let's install all necessary packages

In [ ]:
# ============================================
# STEP 1: Install Required Packages
# ============================================
#
# WHAT WE'RE DOING:
# Installing all Python libraries needed for the pharmaceutical document
# Q&A pipeline: Gradio for the UI, PyMuPDF for PDF parsing, FAISS for
# vector search, Sentence Transformers for embeddings, Google Generative AI
# for the LLM, and LlamaIndex for advanced document processing.
#
# WHY THIS MATTERS:
# Each library handles a distinct part of the pipeline. Without them, we
# cannot extract text from PDFs, generate vector embeddings, or query the
# Gemini model. The -q flag suppresses verbose output so the cell is easier
# to read.
#
# WHAT YOU'LL SEE:
# Brief progress bars as each package installs. If a package is already
# present in your Colab environment it will be skipped automatically.
# ============================================

# Install required packages
!pip install -q gradio
!pip install -q gradio_pdf
!pip install -q pypdf PyPDF2 pymupdf
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q numpy pandas

# Install LlamaIndex packages for enhanced document processing
!pip install -q llama-index
!pip install -q llama-index-readers-file
!pip install -q llama-index-embeddings-huggingface
!pip install -q llama-index-vector-stores-faiss

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 11.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the

In [ ]:
# Install llama-cpp-python with CUDA 12.x support
!pip install --no-cache-dir llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu123

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu123
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.5/444.5 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 175.7 MB/s eta 0:00:00


## 🔧 Core Imports and Configuration

In [ ]:
# ============================================
# STEP 2: Core Imports and Configuration
# ============================================
#
# WHAT WE'RE DOING:
# Importing all libraries, connecting to the Gemini LLM using a Colab
# secret (no hardcoded keys), and loading the all-MiniLM-L6-v2 embedding
# model that will convert text into vector representations.
#
# WHY THIS MATTERS:
# The Gemini model powers document classification and answer generation.
# The embedding model converts text chunks into numerical vectors so we can
# perform semantic similarity search with FAISS. Storing your API key in
# Colab Secrets (not in code) keeps it secure and out of version control.
#
# WHAT YOU'LL SEE:
# A confirmation message once the embedding model is downloaded and loaded.
# If the GEMINI_API_KEY secret is missing you will see an error -- add it
# via the Colab key icon on the left sidebar before proceeding.
# ============================================

import os
import re
import gradio as gr
from gradio_pdf import PDF
import fitz  # PyMuPDF
from PyPDF2 import PdfReader
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
import json
from datetime import datetime
import hashlib

# LlamaIndex imports for enhanced document processing
from llama_cpp import Llama
from llama_index.core import Document, VectorStoreIndex, StorageContext
from llama_index.core.schema import TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator

In [ ]:
# Configure Mistral AI model (open source model)

model_path = "/content/mistral.gguf"  # change this to your model path
if not os.path.exists(model_path):
    !wget https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O {model_path}
    print(f"Model downloaded to {model_path}")

# Verify file exists and check size
if os.path.exists(model_path):
    print(f"Model file exists. Size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")
else:
    print("Model file not found!")

# Load the model with GPU acceleration
try:
    mistral_llm = Llama(
        model_path=model_path,
        n_gpu_layers=1,  # Start with 1 layer on GPU to be safe
        n_ctx=2048,      # Context window size
        verbose=True     # Show loading progress
    )

    print("Model loaded successfully!")



except Exception as e:
    print(f"Error loading model: {e}")

--2026-07-16 19:44:33--  https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf
Resolving huggingface.co (huggingface.co)... 3.171.171.128, 3.171.171.104, 3.171.171.65, ...
Connecting to huggingface.co (huggingface.co)|3.171.171.128|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cas-bridge.xethub.hf.co/xet-bridge-us/65778ac662d3ac1817cc9201/865f5e4682dddb29c2e20270b2471a7590c83a414bbf1d72cf4c08fdff2eeca4?Expires=1784234673&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzY1Nzc4YWM2NjJkM2FjMTgxN2NjOTIwMS84NjVmNWU0NjgyZGRkYjI5YzJlMjAyNzBiMjQ3MWE3NTkwYzgzYTQxNGJiZjFkNzJjZjRjMDhmZGZmMmVlY2E0KiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4NDIzNDY3M319fV19&Signature=MEQCIDEUhv%7E6YU7-90-3bKSotP-pD5fPcsYlp0Ejgcy39Mr%7EAiAVygn0xNNzdonKJpej3Vyp77mgBrIap8amCHkrDKLT7Q__&Key-Pair-Id=K3EPXBYC3CKDRZ&X-Xet-Cas-Uid=public&response-

llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /content/mistral.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.

Model downloaded to /content/mistral.gguf
Model file exists. Size: 4166.07 MB


llm_load_print_meta: freq_scale_train = 1
llm_load_print_meta: n_ctx_orig_yarn  = 32768
llm_load_print_meta: rope_finetuned   = unknown
llm_load_print_meta: ssm_d_conv       = 0
llm_load_print_meta: ssm_d_inner      = 0
llm_load_print_meta: ssm_d_state      = 0
llm_load_print_meta: ssm_dt_rank      = 0
llm_load_print_meta: ssm_dt_b_c_rms   = 0
llm_load_print_meta: model type       = 7B
llm_load_print_meta: model ftype      = Q4_K - Medium
llm_load_print_meta: model params     = 7.24 B
llm_load_print_meta: model size       = 4.07 GiB (4.83 BPW) 
llm_load_print_meta: general.name     = mistralai_mistral-7b-instruct-v0.2
llm_load_print_meta: BOS token        = 1 '<s>'
llm_load_print_meta: EOS token        = 2 '</s>'
llm_load_print_meta: UNK token        = 0 '<unk>'
llm_load_print_meta: PAD token        = 0 '<unk>'
llm_load_print_meta: LF token         = 13 '<0x0A>'
llm_load_print_meta: max token length = 48
ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    yes
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS

Model loaded successfully!


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | AVX512_BF16 = 0 | FMA = 1 | NEON = 0 | SVE = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | MATMUL_INT8 = 0 | LLAMAFILE = 1 | 
Model metadata: {'tokenizer.chat_template': "{{ bos_token }}{% for message in messages %}{% if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}{{ raise_exception('Conversation roles must alternate user/assistant/user/assistant/...') }}{% endif %}{% if message['role'] == 'user' %}{{ '[INST] ' + message['content'] + ' [/INST]' }}{% elif message['role'] == 'assistant' %}{{ message['content'] + eos_token}}{% else %}{{ raise_exception('Only user and assistant roles are supported!') }}{% endif %}{% endfor %}", 'tokenizer.ggml.add_eos_token': 'false', 'tokenizer.ggml.padding_token_id': '0', 'tokenizer.ggml.unknown_token_id': '0', 'tokenizer.ggml.eos_token_id': '2', 'general.architecture': 'llama', 'llama.rope.freq_base': 

In [ ]:
def call_mistral(prompt, max_tokens=300, temperature=0.0):
    """
    Calls the local Mistral model using llama-cpp-python.
    This replaces Gemini's generate_content() calls.
    """

    formatted_prompt = f"""
[INST]
{prompt}
[/INST]
"""

    response = mistral_llm(
        formatted_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        stop=["</s>", "[/INST]"]
    )

    return response["choices"][0]["text"].strip()

In [ ]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import login

login()

In [ ]:
import os

# os.environ["HF_TOKEN"] = "your_huggingface_token_here"
os.environ["HF_TOKEN"] = "your_hugging_face_token"

In [ ]:
# Initialize embedding models (both for compatibility)
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# llama_embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 📄 Data Structures for Enhanced Document Management
Let's define our data structures to handle complex document metadata:

In [ ]:
# ============================================
# STEP 3: Data Structures for Document Management
# ============================================
#
# WHAT WE'RE DOING:
# Defining three Python dataclasses that act as structured containers for
# the information we extract from the PDF: individual page data, logical
# document groupings, and rich chunk metadata.
#
# WHY THIS MATTERS:
# A pharmaceutical blob PDF may contain many distinct documents -- a Cover
# Letter, a Certificate of Quality, a Packaging Specification, etc. -- all
# merged into one file. These dataclasses let us track which pages belong
# to which document and carry that metadata through the entire pipeline so
# answers can cite exact page ranges and document types.
#
# WHAT YOU'LL SEE:
# No output. The classes are defined silently and will be used in later
# cells when we process the uploaded PDF.
# ============================================

@dataclass
class PageInfo:
    """Stores information about a single page"""
    page_num: int
    text: str
    doc_type: Optional[str] = None
    page_in_doc: int = 0

@dataclass
class LogicalDocument:
    """Represents a logical document within a PDF"""
    doc_id: str
    doc_type: str
    page_start: int
    page_end: int
    text: str
    chunks: List[Dict] = None

@dataclass
class ChunkMetadata:
    """Rich metadata for each chunk"""
    chunk_id: str
    doc_id: str
    doc_type: str
    chunk_index: int
    page_start: int
    page_end: int
    text: str
    embedding: Optional[np.ndarray] = None

## 🧠 Document Intelligence Functions
These functions handle document classification and boundary detection:

In [ ]:
# ============================================
# STEP 4: Document Intelligence Functions
# ============================================
#
# WHAT WE'RE DOING:
# Defining two key functions: classify_document_type (uses Mistral to
# identify which of the 7 pharmaceutical document types a page belongs to)
# and detect_document_boundary (uses Mistral to decide whether two
# consecutive pages are part of the same document or two different ones).
# A helper clean_doc_type() normalises the LLM's free-text response into
# one of the valid labels.
#
# WHY THIS MATTERS:
# A pharmaceutical blob PDF is a single file that bundles many different
# document types together. Before we can answer questions accurately, we
# must split it into logical segments so retrieval can be scoped to the
# right document. For example, a question about lot numbers should pull
# from the Certificate of Quality, not the Cover Letter.
#
# WHAT YOU'LL SEE:
# No output when this cell runs. The functions are called automatically
# during PDF processing in Step 6.
# ============================================

VALID_DOC_TYPES = [
    "Cover Letter", "Certificate Of Quality", "Packaging Specification",
    "Bse/Tse Declaration", "Material Description", "Supplier Qualification",
    "Chain Of Custody", "Other"
]

def clean_doc_type(response):
    """Clean up LLM response to extract a valid doc_type label."""
    cleaned = response.strip().replace('"', '').replace('`', '').replace('*', '').lower().replace(".", "").strip()
    cleaned_title = cleaned.title()

    for label in VALID_DOC_TYPES:
        if label.lower() in cleaned.lower():
            return label

    return cleaned_title


def classify_document_type(text: str, max_length: int = 1500) -> str:
    """
    Classify the document type based on its content using local Mistral.
    """

    text_sample = text[:max_length] if len(text) > max_length else text

    prompt = f"""You are a pharmaceutical document classifier. Based on the page
content below, classify it into ONE of these document types:

- Cover Letter: A formal letter, often starting with "To Whom It May Concern", discussing product information or storage conditions.
- Certificate Of Quality: Contains lot numbers, manufacture dates, expiration dates, and test results.
- Packaging Specification: Describes packaging components, materials, part numbers, and configuration change history.
- Bse/Tse Declaration: A declaration about animal-origin materials and transmissible spongiform encephalopathy compliance.
- Material Description: Lists materials of construction, sterilization compatibility, and physical properties.
- Supplier Qualification: Contains supplier audit history, certifications, and approved product lists.
- Chain Of Custody: Lists manufactured assemblies, traceability information, and manufacturing-to-shipment flow.
- Other: Use only if the content does not match any of the above.

Page content:
{text_sample}

Respond with ONLY the document type name. No explanation."""

    try:
        response_text = call_mistral(prompt, max_tokens=50)
        return clean_doc_type(response_text)

    except Exception as e:
        print(f"Classification error: {e}")
        return "Other"


def detect_document_boundary(prev_text: str, curr_text: str, current_doc_type: str = None) -> bool:
    """
    Detect if two consecutive pages belong to the same document.
    Returns True if they are from the same document.
    """

    if not prev_text or not curr_text:
        return False

    prev_sample = prev_text[-500:] if len(prev_text) > 500 else prev_text
    curr_sample = curr_text[:500] if len(curr_text) > 500 else curr_text

    prompt = f"""Determine if these two pages are from the SAME pharmaceutical document.

Current document type: {current_doc_type or 'Unknown'}

A NEW document starts when the page has:
- A different document title or heading
- A completely different topic or subject matter
- Its own header with a new document number or reference

Pages belong to the SAME document when:
- The second page says "continued" or "page 2 of 2"
- The content directly continues the previous page's discussion
- They share the same document number or title

End of Previous Page:
...{prev_sample}

Start of Current Page:
{curr_sample}...

Answer ONLY 'Yes' if same document or 'No' if different document."""

    try:
        response_text = call_mistral(prompt, max_tokens=20)
        return response_text.strip().lower().startswith("yes")

    except Exception as e:
        print(f"Boundary detection error: {e}")
        return True

## 📑 Advanced PDF Processing Pipeline
Now let's build the enhanced PDF processing pipeline:

In [ ]:
# ============================================
# STEP 5: Advanced PDF Processing Pipeline
# ============================================
#
# WHAT WE'RE DOING:
# Defining extract_and_analyze_pdf(), which opens the uploaded PDF page by
# page, extracts text (with OCR fallback for scanned pages), calls
# classify_document_type() on the first page of each segment, and uses
# detect_document_boundary() to decide where one pharmaceutical document
# ends and the next begins.
#
# WHY THIS MATTERS:
# A pharmaceutical blob PDF is not a single document -- it is many
# documents concatenated together. This function produces two outputs:
# (1) a list of PageInfo objects (one per page) and (2) a list of
# LogicalDocument objects (one per identified sub-document). Accurate
# boundary detection is critical: if pages are grouped incorrectly,
# retrieved chunks will mix content from unrelated documents.
#
# WHAT YOU'LL SEE:
# No output when this cell runs. It is called inside process_pdf() in the
# EnhancedDocumentStore class (Step 8) when the user uploads a file.
# ============================================

def extract_and_analyze_pdf(pdf_file) -> Tuple[List[PageInfo], List[LogicalDocument]]:
    """
    Extract text from PDF and perform intelligent document analysis.
    Returns both page-level info and logical document groupings.
    Supports various file types including scanned PDFs with OCR.
    """
    print("Starting PDF extraction and analysis...")

    # Extract text from each page
    if isinstance(pdf_file, dict) and "content" in pdf_file:
        doc = fitz.open(stream=pdf_file["content"], filetype="pdf")
    elif hasattr(pdf_file, "read"):
        doc = fitz.open(stream=pdf_file.read(), filetype="pdf")
    else:
        doc = fitz.open(pdf_file)

    pages_info = []
    for i, page in enumerate(doc):
        text = page.get_text()

        # If no text found, try OCR (for scanned documents)
        if not text.strip():
            print(f"  Page {i}: No text found, attempting OCR...")
            try:
                # Convert page to image and perform OCR
                pix = page.get_pixmap()
                img_data = pix.tobytes("png")
                from PIL import Image
                import pytesseract
                import io

                img = Image.open(io.BytesIO(img_data))
                text = pytesseract.image_to_string(img)
                print(f"  Page {i}: OCR extracted {len(text)} characters")
            except Exception as e:
                print(f"  Page {i}: OCR failed - {e}")
                text = ""

        pages_info.append(PageInfo(page_num=i, text=text))

    doc.close()

    if not pages_info:
        raise ValueError("No text could be extracted from PDF")

    print(f"Extracted {len(pages_info)} pages")

    # Perform document classification and boundary detection
    print("Analyzing document structure...")
    logical_docs = []
    current_doc_type = None
    current_doc_pages = []
    doc_counter = 0

    for i, page_info in enumerate(pages_info):
        if i == 0:
            # First page - classify document type
            current_doc_type = classify_document_type(page_info.text)
            page_info.doc_type = current_doc_type
            page_info.page_in_doc = 0
            current_doc_pages = [page_info]
            print(f"  Page {i}: New document detected - {current_doc_type}")
        else:
            # Check if this page continues the previous document
            prev_text = pages_info[i-1].text
            is_same = detect_document_boundary(prev_text, page_info.text, current_doc_type)

            if is_same:
                # Continue current document
                page_info.doc_type = current_doc_type
                page_info.page_in_doc = len(current_doc_pages)
                current_doc_pages.append(page_info)
            else:
                # New document detected - save previous and start new
                logical_doc = LogicalDocument(
                    doc_id=f"doc_{doc_counter}",
                    doc_type=current_doc_type,
                    page_start=current_doc_pages[0].page_num,
                    page_end=current_doc_pages[-1].page_num,
                    text="\n\n".join([p.text for p in current_doc_pages])
                )
                logical_docs.append(logical_doc)
                doc_counter += 1

                # Start new document
                current_doc_type = classify_document_type(page_info.text)
                page_info.doc_type = current_doc_type
                page_info.page_in_doc = 0
                current_doc_pages = [page_info]
                print(f"  Page {i}: New document detected - {current_doc_type}")

    # Don't forget the last document
    if current_doc_pages:
        logical_doc = LogicalDocument(
            doc_id=f"doc_{doc_counter}",
            doc_type=current_doc_type,
            page_start=current_doc_pages[0].page_num,
            page_end=current_doc_pages[-1].page_num,
            text="\n\n".join([p.text for p in current_doc_pages])
        )
        logical_docs.append(logical_doc)

    print(f"Identified {len(logical_docs)} logical documents")
    for ld in logical_docs:
        print(f"   - {ld.doc_type}: Pages {ld.page_start}-{ld.page_end}")

    return pages_info, logical_docs

## ✂️ Intelligent Chunking with Metadata Preservation
We'll provide two chunking approaches - our custom implementation and LlamaIndex's built-in capabilities:

In [ ]:
# ============================================
# STEP 6: Intelligent Chunking with Metadata Preservation
# ============================================
#
# WHAT WE'RE DOING:
# Breaking each LogicalDocument into smaller overlapping text chunks and
# attaching rich metadata (document type, page range, chunk index) to each
# one. Two approaches are provided: a custom sliding-window chunker and a
# LlamaIndex SentenceSplitter that respects sentence boundaries.
#
# WHY THIS MATTERS:
# Language models and vector search work best with short focused text
# segments rather than entire documents. Overlapping chunks (100-word
# overlap) prevent important information from being split across chunk
# boundaries. Preserving metadata on every chunk ensures that when a chunk
# is retrieved, we can tell the user exactly which document type and page
# it came from -- essential for pharmaceutical compliance traceability.
#
# WHAT YOU'LL SEE:
# No output when this cell runs. When process_all_documents() is called
# during PDF processing you will see lines like:
#   Certificate Of Quality: Created 3 chunks
#   Packaging Specification: Created 2 chunks
# ============================================

def chunk_document_with_metadata(logical_doc: LogicalDocument,
                                chunk_size: int = 500,
                                overlap: int = 100) -> List[ChunkMetadata]:
    """
    Chunk a logical document while preserving rich metadata.
    Uses sliding window with overlap for better context.
    """
    chunks_metadata = []
    words = logical_doc.text.split()

    if len(words) <= chunk_size:
        # Document is small enough to be a single chunk
        chunk_meta = ChunkMetadata(
            chunk_id=f"{logical_doc.doc_id}_chunk_0",
            doc_id=logical_doc.doc_id,
            doc_type=logical_doc.doc_type,
            chunk_index=0,
            page_start=logical_doc.page_start,
            page_end=logical_doc.page_end,
            text=logical_doc.text
        )
        chunks_metadata.append(chunk_meta)
    else:
        # Create overlapping chunks
        stride = chunk_size - overlap
        for i, start_idx in enumerate(range(0, len(words), stride)):
            end_idx = min(start_idx + chunk_size, len(words))
            chunk_text = ' '.join(words[start_idx:end_idx])

            # Calculate which pages this chunk spans
            # (simplified - in production, track more precisely)
            chunk_position = start_idx / len(words)
            page_range = logical_doc.page_end - logical_doc.page_start
            relative_page = int(chunk_position * page_range)
            chunk_page_start = logical_doc.page_start + relative_page
            chunk_page_end = min(chunk_page_start + 1, logical_doc.page_end)

            chunk_meta = ChunkMetadata(
                chunk_id=f"{logical_doc.doc_id}_chunk_{i}",
                doc_id=logical_doc.doc_id,
                doc_type=logical_doc.doc_type,
                chunk_index=i,
                page_start=chunk_page_start,
                page_end=chunk_page_end,
                text=chunk_text
            )
            chunks_metadata.append(chunk_meta)

            if end_idx >= len(words):
                break

    return chunks_metadata

def chunk_with_llama_index(logical_doc: LogicalDocument,
                           chunk_size: int = 500,
                           chunk_overlap: int = 100) -> List[Document]:
    """
    Alternative: Use LlamaIndex's advanced chunking with metadata.
    """
    # Create LlamaIndex document with metadata
    doc = Document(
        text=logical_doc.text,
        metadata={
            "doc_id": logical_doc.doc_id,
            "doc_type": logical_doc.doc_type,
            "page_start": logical_doc.page_start,
            "page_end": logical_doc.page_end,
            "source": f"{logical_doc.doc_type}_document"
        }
    )

    # Use LlamaIndex's sentence splitter for better chunking
    splitter = SentenceSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        paragraph_separator="\n\n",
        separator=" ",
    )

    # Create nodes (chunks) from document
    nodes = splitter.get_nodes_from_documents([doc])

    # Convert to our ChunkMetadata format for consistency
    chunks_metadata = []
    for i, node in enumerate(nodes):
        chunk_meta = ChunkMetadata(
            chunk_id=f"{logical_doc.doc_id}_chunk_{i}",
            doc_id=logical_doc.doc_id,
            doc_type=logical_doc.doc_type,
            chunk_index=i,
            page_start=node.metadata.get("page_start", logical_doc.page_start),
            page_end=node.metadata.get("page_end", logical_doc.page_end),
            text=node.text
        )
        chunks_metadata.append(chunk_meta)

    return chunks_metadata

def process_all_documents(logical_docs: List[LogicalDocument],
                         use_llama_index: bool = False) -> List[ChunkMetadata]:
    """
    Process all logical documents into chunks with metadata.
    Can use either custom or LlamaIndex chunking.
    """
    all_chunks = []

    for logical_doc in logical_docs:
        if use_llama_index:
            chunks = chunk_with_llama_index(logical_doc)
        else:
            chunks = chunk_document_with_metadata(logical_doc)

        logical_doc.chunks = chunks  # Store reference
        all_chunks.extend(chunks)
        print(f"  {logical_doc.doc_type}: Created {len(chunks)} chunks")

    return all_chunks

## 🎯 Query Routing and Intelligent Retrieval

In [ ]:
# ============================================
# STEP 7: Query Routing and Intelligent Retrieval
# ============================================
#
# WHAT WE'RE DOING:
# Defining predict_query_document_type() which asks Gemini to guess which
# of the 7 pharmaceutical document types is most likely to contain the
# answer to the user's question. Then defining IntelligentRetriever, which
# builds per-document-type FAISS indices and uses the query routing
# prediction to search the most relevant index first.
#
# WHY THIS MATTERS:
# Searching all chunks equally would mix results from a Cover Letter with
# results from a Certificate of Quality. By routing "What is the lot
# number?" to the Certificate Of Quality index and "Who is the supplier?"
# to Supplier Qualification, we improve both precision and speed. When
# routing confidence is low (<0.7) the system falls back to a global search
# across all document types.
#
# WHAT YOU'LL SEE:
# No output when this cell runs. During a query you will see a line like:
#   Query routed to: Certificate Of Quality (confidence: 0.91)
# ============================================


def predict_query_document_type(query: str) -> Tuple[str, float]:
    """
    Predict which pharmaceutical document type is most likely to contain
    the answer. Uses local Mistral instead of Gemini.
    """

    prompt = f"""Analyze this query and predict which pharmaceutical document type
would most likely contain the answer.

Query: "{query}"

Choose the MOST LIKELY type from:
- Cover Letter: Formal letters about product information or storage conditions
- Certificate Of Quality: Lot numbers, manufacture/expiration dates, test results
- Packaging Specification: Packaging components, materials, part numbers
- Bse/Tse Declaration: Animal-origin material declarations, TSE compliance
- Material Description: Materials of construction, sterilization compatibility
- Supplier Qualification: Supplier audits, ISO certifications, approved products
- Chain Of Custody: Manufactured assemblies, traceability, shipment flow
- Other: General or unclear queries

Respond in JSON format only:
{{"type": "DocumentType", "confidence": 0.85}}

Confidence should be between 0.0 and 1.0."""

    try:
        response_text = call_mistral(prompt, max_tokens=120)

        json_match = re.search(r"\{.*?\}", response_text, re.DOTALL)

        if json_match:
            result = json.loads(json_match.group(0))
            predicted = result.get("type", "Other")
            confidence = float(result.get("confidence", 0.5))
            return clean_doc_type(predicted), confidence

        predicted = clean_doc_type(response_text)

        if predicted in VALID_DOC_TYPES:
            return predicted, 0.6

        return "Other", 0.0

    except Exception as e:
        print(f"Query routing error: {e}")
        return "Other", 0.0


class IntelligentRetriever:
    """
    Fixed retrieval system with:
    1. Cosine similarity instead of raw L2 distance
    2. Normalized embeddings
    3. Safe top-k handling
    4. Filtering of invalid FAISS -1 results
    5. Duplicate chunk removal
    """

    def __init__(self):
        self.index = None
        self.chunks_metadata = []
        self.doc_type_indices = {}

    def build_indices(self, chunks_metadata: List[ChunkMetadata]):
        """
        Build FAISS indices with document type segregation.
        Uses normalized embeddings + inner product, which gives cosine similarity.
        """

        print("Building vector indices...")

        self.chunks_metadata = chunks_metadata
        self.doc_type_indices = {}

        texts = [chunk.text for chunk in chunks_metadata]

        embeddings = embed_model.encode(
            texts,
            show_progress_bar=True,
            normalize_embeddings=True
        )

        embeddings = np.asarray(embeddings).astype("float32")

        for i, chunk in enumerate(chunks_metadata):
            chunk.embedding = embeddings[i]

        dim = embeddings.shape[1]

        # Main global cosine-similarity index
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(embeddings)

        # Separate cosine-similarity indices by document type
        doc_types = set(chunk.doc_type for chunk in chunks_metadata)

        for doc_type in doc_types:
            type_indices = [
                i for i, chunk in enumerate(chunks_metadata)
                if chunk.doc_type == doc_type
            ]

            if type_indices:
                type_embeddings = embeddings[type_indices].astype("float32")

                type_index = faiss.IndexFlatIP(dim)
                type_index.add(type_embeddings)

                self.doc_type_indices[doc_type] = {
                    "index": type_index,
                    "mapping": type_indices
                }

        print(f"Indexed {len(chunks_metadata)} chunks across {len(doc_types)} document types")

    def _search_index(self, index, mapping, query_embedding, k):
        """
        Safely search a FAISS index.

        Important fix:
        FAISS returns -1 when k is larger than the number of available vectors.
        This function prevents -1 from being mapped to the last chunk.
        """

        if index is None or index.ntotal == 0:
            return []

        # Do not ask FAISS for more results than the index contains
        safe_k = min(int(k), index.ntotal)

        scores, indices = index.search(query_embedding, safe_k)

        results = []
        seen_chunk_ids = set()
        seen_texts = set()

        for faiss_idx, score in zip(indices[0], scores[0]):
            faiss_idx = int(faiss_idx)

            # Skip invalid FAISS results
            if faiss_idx < 0:
                continue

            original_chunk_idx = mapping[faiss_idx]
            chunk = self.chunks_metadata[original_chunk_idx]

            # Avoid duplicate chunk IDs
            if chunk.chunk_id in seen_chunk_ids:
                continue

            # Avoid exact duplicate text
            normalized_text = " ".join(chunk.text.split()).strip().lower()
            if normalized_text in seen_texts:
                continue

            seen_chunk_ids.add(chunk.chunk_id)
            seen_texts.add(normalized_text)

            # Because we use normalized embeddings + IndexFlatIP,
            # score is cosine similarity.
            score = float(score)

            # Clamp score into readable 0-1 range
            score = max(0.0, min(1.0, score))

            results.append((chunk, score))

        return results

    def retrieve(
        self,
        query: str,
        k: int = 4,
        filter_doc_type: Optional[str] = None,
        auto_route: bool = True
    ) -> List[Tuple[ChunkMetadata, float]]:
        """
        Retrieve relevant chunks with optional filtering and routing.
        Returns chunks with cosine similarity scores.
        """

        query_embedding = embed_model.encode(
            [query],
            normalize_embeddings=True
        )

        query_embedding = np.asarray(query_embedding).astype("float32")

        # Case 1: Manual document-type filter selected
        if filter_doc_type and filter_doc_type in self.doc_type_indices:
            print(f"Using manual filter: {filter_doc_type}")

            type_data = self.doc_type_indices[filter_doc_type]

            return self._search_index(
                index=type_data["index"],
                mapping=type_data["mapping"],
                query_embedding=query_embedding,
                k=k
            )

        # Case 2: Auto-route query to predicted document type
        if auto_route:
            predicted_type, confidence = predict_query_document_type(query)

            print(f"Query routed to: {predicted_type} (confidence: {confidence:.2f})")

            if confidence > 0.7 and predicted_type in self.doc_type_indices:
                type_data = self.doc_type_indices[predicted_type]

                return self._search_index(
                    index=type_data["index"],
                    mapping=type_data["mapping"],
                    query_embedding=query_embedding,
                    k=k
                )

        # Case 3: Fallback to global search
        print("Using global search across all document types")

        global_mapping = list(range(len(self.chunks_metadata)))

        return self._search_index(
            index=self.index,
            mapping=global_mapping,
            query_embedding=query_embedding,
            k=k
        )

## 💬 Enhanced Answer Generation with Source Attribution

In [ ]:
# ============================================
# STEP 8: Enhanced Answer Generation with Source Attribution
# ============================================
#
# WHAT WE'RE DOING:
# Defining generate_answer_with_sources(), which takes the top-k retrieved
# chunks and sends them to Gemini as context, asking it to answer the user's
# question using only that context. The function returns the answer text, a
# list of source citations (document type, page range, relevance score),
# and an overall confidence score derived from the retrieval scores.
#
# WHY THIS MATTERS:
# Pharmaceutical documentation must be traceable. Simply generating an
# answer without citing which document and page it came from would not meet
# compliance standards. By injecting "[From Certificate Of Quality, Pages
# 2-3]" labels into the context prompt, we encourage Gemini to ground its
# answer in the actual source material and to tell the user where to verify
# the information.
#
# WHAT YOU'LL SEE:
# No output when this cell runs. After a user submits a question in the
# Gradio interface the answer will appear in the chat panel along with
# source citations and a confidence percentage.
# ============================================

def generate_answer_with_sources(query: str, retrieved_chunks: List[Tuple[ChunkMetadata, float]]) -> Dict:
    """
    Generate answer with detailed source attribution using local Mistral.
    """

    if not retrieved_chunks:
        return {
            "answer": "I couldn't find relevant information to answer your question.",
            "sources": [],
            "confidence": 0.0
        }

    context_parts = []
    sources = []

    for chunk_meta, score in retrieved_chunks:
        context_parts.append(
            f"[From {chunk_meta.doc_type}, Pages {chunk_meta.page_start}-{chunk_meta.page_end}]"
        )
        context_parts.append(chunk_meta.text)
        context_parts.append("")

        sources.append({
            "doc_type": chunk_meta.doc_type,
            "pages": f"{chunk_meta.page_start}-{chunk_meta.page_end}",
            "relevance": f"{score:.2%}",
            "preview": chunk_meta.text[:100] + "..."
        })

    context = "\n".join(context_parts)

    print("\n===== RETRIEVED CHUNKS SENT TO MISTRAL =====\n")

    for i, (chunk_meta, score) in enumerate(retrieved_chunks, start=1):
        print(f"Chunk {i}")
        print(f"Document Type: {chunk_meta.doc_type}")
        print(f"Pages: {chunk_meta.page_start}-{chunk_meta.page_end}")
        print(f"Relevance Score: {score:.4f}")
        print("Text Preview:")
        print(chunk_meta.text[:1000])
        print("-" * 80)

    # Keep context short enough for local Mistral
    context = context[:3500]

    prompt = f"""You are answering questions about pharmaceutical documentation,
including certificates of quality, packaging specifications, and compliance declarations.

Use the provided context to answer the question accurately.
Be specific and cite which document type and pages support your answer.

Context:
{context}

Question:
{query}

Instructions:
1. Answer based ONLY on the provided context.
2. Mention which document type and page range contain the information.
3. Be concise but complete.
4. If the context does not contain enough information, say so.

Answer:"""

    try:
        answer = call_mistral(prompt, max_tokens=350)

        avg_score = sum(s for _, s in retrieved_chunks) / len(retrieved_chunks)

        return {
            "answer": answer,
            "sources": sources,
            "confidence": avg_score,
            "chunks_used": len(retrieved_chunks)
        }

    except Exception as e:
        print(f"Answer generation error: {e}")

        return {
            "answer": f"Error generating answer: {str(e)}",
            "sources": sources,
            "confidence": 0.0
        }

## 🏗️ Enhanced Document Store

In [ ]:
# ============================================
# STEP 9: Enhanced Document Store
# ============================================
#
# WHAT WE'RE DOING:
# Defining EnhancedDocumentStore, the central orchestrator class that ties
# together all previous steps: PDF extraction, logical document grouping,
# chunking, index building, and query handling. A single global instance
# (doc_store) is created and shared across the Gradio UI callbacks.
#
# WHY THIS MATTERS:
# The document store acts as the in-memory database for the entire session.
# It holds all page data, logical documents, chunk metadata, and the FAISS
# indices. Every Gradio interaction (upload, question, clear) goes through
# this class, which keeps state management simple and avoids re-processing
# the PDF on every query.
#
# WHAT YOU'LL SEE:
# No output when this cell runs. When the user uploads a pharmaceutical
# blob PDF via the Gradio interface, doc_store.process_pdf() is called and
# prints progress lines for each processing stage.
# ============================================

class EnhancedDocumentStore:
    """
    Manages the complete document processing and retrieval pipeline.
    """

    def __init__(self):
        self.pages_info = []
        self.logical_docs = []
        self.chunks_metadata = []
        self.retriever = IntelligentRetriever()
        self.is_ready = False
        self.processing_stats = {}
        self.filename = None

    def process_pdf(self, pdf_file, filename: str = "document.pdf"):
        """
        Complete PDF processing pipeline.
        """
        self.filename = filename
        self.is_ready = False
        start_time = datetime.now()

        try:
            # Extract and analyze PDF
            self.pages_info, self.logical_docs = extract_and_analyze_pdf(pdf_file)

            # Chunk documents with metadata
            self.chunks_metadata = process_all_documents(self.logical_docs)

            # Build retrieval indices
            self.retriever.build_indices(self.chunks_metadata)

            # Calculate processing statistics
            process_time = (datetime.now() - start_time).total_seconds()
            self.processing_stats = {
                'filename': filename,
                'total_pages': len(self.pages_info),
                'documents_found': len(self.logical_docs),
                'total_chunks': len(self.chunks_metadata),
                'document_types': list(set(doc.doc_type for doc in self.logical_docs)),
                'processing_time': f"{process_time:.1f}s"
            }

            self.is_ready = True
            return True, self.processing_stats

        except Exception as e:
            return False, {'error': str(e)}

    def query(self, question: str, filter_type: Optional[str] = None,
             auto_route: bool = True, k: int = 4) -> Dict:
        """
        Query the document store.
        """
        if not self.is_ready:
            return {
                'answer': "Please upload and process a PDF first.",
                'sources': [],
                'confidence': 0.0
            }

        # Retrieve relevant chunks
        retrieved = self.retriever.retrieve(
            question, k=k,
            filter_doc_type=filter_type,
            auto_route=auto_route
        )

        # Generate answer with sources
        result = generate_answer_with_sources(question, retrieved)
        result['filter_used'] = filter_type or ('auto' if auto_route else 'none')

        return result

    def get_document_structure(self) -> List[Dict]:
        """
        Get the document structure for UI display.
        """
        if not self.logical_docs:
            return []

        structure = []
        for doc in self.logical_docs:
            structure.append({
                'id': doc.doc_id,
                'type': doc.doc_type,
                'pages': f"{doc.page_start + 1}-{doc.page_end + 1}",  # 1-indexed for UI
                'chunks': len(doc.chunks) if doc.chunks else 0,
                'preview': doc.text[:200] + "..." if len(doc.text) > 200 else doc.text
            })

        return structure

## 🎨 Gradio Interface with Enhanced Features
Now let's create the sophisticated Gradio interface:

In [ ]:
# ============================================
# STEP 10: Gradio Interface
# ============================================
#
# WHAT WE'RE DOING:
# Building the Gradio web UI that ties all pipeline components together.
# The interface has three columns: a PDF upload on the left, document info
# and retrieval settings in the middle, and a chat panel on the right.
# Users upload pharma-blob-sample.pdf, the system processes it, detects
# document boundaries, builds the vector index, and then allows natural
# language Q&A against the identified pharmaceutical sub-documents.
#
# WHY THIS MATTERS:
# The Gradio interface makes the RAG pipeline accessible to non-technical
# users -- a pharmaceutical quality engineer should be able to ask
# "What is the expiration date on the Certificate of Quality?" without
# needing to understand FAISS or embeddings. The document type filter
# dropdown and auto-route toggle give advanced users fine-grained control
# over retrieval scope.
#
# WHAT YOU'LL SEE:
# A browser tab (or inline Colab iframe) showing the full Q&A interface.
# Upload pharma-blob-sample.pdf, wait for processing to complete, then
# type questions in the chat panel on the right.
# ============================================

# Global store instance
doc_store = EnhancedDocumentStore()

def process_pdf_handler(pdf_file):
    """Handle PDF upload and processing."""
    if pdf_file is None:
        return "Please upload a PDF file", None, gr.update(choices=["All"])

    # Process the PDF
    success, stats = doc_store.process_pdf(pdf_file,
                                          filename=pdf_file.split('/')[-1] if isinstance(pdf_file, str) else
getattr(pdf_file, 'name', 'pharma-blob-sample.pdf'))

    if success:
        # Prepare status message
        status_msg = f"""
**Successfully Processed:**
- File: {stats['filename']}
- Pages: {stats['total_pages']}
- Documents Found: {stats['documents_found']}
- Chunks Created: {stats['total_chunks']}
- Types: {', '.join(stats['document_types'])}
- Time: {stats['processing_time']}
"""

        # Get document structure for display
        structure = doc_store.get_document_structure()
        structure_display = "\n".join([
            f"- **{doc['type']}** (Pages {doc['pages']}): {doc['chunks']} chunks"
            for doc in structure
        ])

        # Update filter choices
        doc_types = ["All"] + stats['document_types']

        return status_msg, structure_display, gr.update(choices=doc_types, value="All")
    else:
        return f"Error: {stats.get('error', 'Unknown error')}", None, gr.update(choices=["All"])

def chat_handler(message, history, doc_filter, auto_route, num_chunks):
    """Handle chat interactions."""
    if not doc_store.is_ready:
        response = "Please upload and process a pharmaceutical PDF document first."
        return history + [{"role": "user", "content": message}, {"role": "assistant", "content": response}]

    # Query the document store
    filter_type = None if doc_filter == "All" else doc_filter
    result = doc_store.query(
        message,
        filter_type=filter_type,
        auto_route=auto_route and filter_type is None,
        k=num_chunks
    )

    # Format response with sources
    response = f"{result['answer']}\n\n"

    if result['sources']:
        response += "**Sources:**\n"
        for src in result['sources']:
            response += f"- {src['doc_type']} (Pages {src['pages']}) - Relevance: {src['relevance']}\n"

    response += f"\n*Confidence: {result['confidence']:.1%} | Filter: {result['filter_used']}*"

    return history + [{"role": "user", "content": message}, {"role": "assistant", "content": response}]

def create_interface():
    """Create the Gradio interface for pharmaceutical document Q&A."""

    with gr.Blocks(title="Pharmaceutical Document Q&A System") as demo:
        gr.Markdown("""
        # Pharmaceutical Document Q&A System
        ### Intelligent Multi-Document Analysis with Advanced RAG Pipeline
        Upload a pharmaceutical blob PDF (e.g. pharma-blob-sample.pdf) to identify
        document types, build a searchable index, and ask questions in natural language.
        """)

        with gr.Row():
            # Left side - PDF upload
            with gr.Column(scale=2):
                pdf_input = gr.File(
                    label="Upload Pharmaceutical PDF",
                    file_types=[".pdf"],
                    type="filepath"
                )

                with gr.Row():
                    process_btn = gr.Button(
                        "Process Document",
                        variant="primary",
                        size="lg",
                        scale=2
                    )
                    clear_all_btn = gr.Button(
                        "Clear All",
                        variant="secondary",
                        size="lg",
                        scale=1
                    )

            # Middle - Document info and settings
            with gr.Column(scale=1):
                gr.Markdown("### Document Info")
                status_output = gr.Markdown(
                    value="Waiting for PDF upload..."
                )

                structure_output = gr.Markdown(
                    value="",
                    label="Document Structure"
                )

                gr.Markdown("### Retrieval Settings")

                doc_filter = gr.Dropdown(
                    choices=["All"],
                    value="All",
                    label="Document Type Filter",
                    info="Filter search to a specific pharmaceutical document type"
                )

                auto_route = gr.Checkbox(
                    value=True,
                    label="Auto-Route Queries",
                    info="Automatically detect the most relevant document type"
                )

                num_chunks = gr.Slider(
                    minimum=1,
                    maximum=10,
                    value=4,
                    step=1,
                    label="Chunks to Retrieve"
                )

            # Right side - Chat interface
            with gr.Column(scale=2):
                gr.Markdown("### Ask Questions")
                chatbot = gr.Chatbot(
                    label="Conversation",
                    height=500,
                    elem_id="chatbot",
                    show_label=False,
                )

                with gr.Row():
                    msg_input = gr.Textbox(
                        label="Ask a question",
                        placeholder="e.g., What is the lot number? What sterilization method was used?",
                        scale=4,
                        show_label=False
                    )
                    send_btn = gr.Button("Send", scale=1, variant="primary")

                with gr.Row():
                    clear_chat_btn = gr.Button("Clear Chat", size="sm", scale=1)
                    example_btn1 = gr.Button("Summarise this document", size="sm", scale=1)
                    example_btn2 = gr.Button("Find lot numbers", size="sm", scale=1)

        # Status bar at the bottom
        with gr.Row():
            status_bar = gr.Markdown(
                value="**Status:** Ready | **Documents:** 0 | **Chunks:** 0",
                elem_id="status_bar"
            )

        # Event handlers
        def update_status_bar():
            """Update the status bar with current statistics."""
            if doc_store.is_ready:
                stats = doc_store.processing_stats
                return (
                    f"**Status:** Ready | "
                    f"**Documents:** {stats.get('documents_found', 0)} | "
                    f"**Chunks:** {stats.get('total_chunks', 0)}"
                )
            return "**Status:** Ready | **Documents:** 0 | **Chunks:** 0"

        def clear_all():
            """Clear everything and reset the interface."""
            global doc_store
            doc_store = EnhancedDocumentStore()
            return (
                None,  # pdf_input
                "Waiting for PDF upload...",  # status_output
                "",  # structure_output
                gr.update(choices=["All"], value="All"),  # doc_filter
                [],  # chatbot
                "",  # msg_input
                update_status_bar()  # status_bar
            )

        # Process PDF handler with status bar update
        def process_pdf_with_status(pdf_file):
            status, structure, filter_update = process_pdf_handler(pdf_file)
            status_bar_text = update_status_bar()
            return status, structure, filter_update, status_bar_text

        # Chat handler with status bar update
        def chat_with_status(message, history, doc_filter, auto_route, num_chunks):
            new_history = chat_handler(message, history, doc_filter, auto_route, num_chunks)
            status_bar_text = update_status_bar()
            return new_history, status_bar_text

        # Example question handlers
        def ask_summary(history):
            return chat_handler(
                "Can you provide a summary of the main points in this document?",
                history, doc_filter.value, auto_route.value, num_chunks.value
            )

        def ask_lot_numbers(history):
            return chat_handler(
                "What lot numbers or batch numbers are mentioned in these documents?",
                history, doc_filter.value, auto_route.value, num_chunks.value
            )

        # Wire up all the events
        process_btn.click(
            fn=process_pdf_with_status,
            inputs=[pdf_input],
            outputs=[status_output, structure_output, doc_filter, status_bar]
        )

        clear_all_btn.click(
            fn=clear_all,
            outputs=[pdf_input, status_output, structure_output, doc_filter,
                    chatbot, msg_input, status_bar]
        )

        # Chat interactions
        msg_input.submit(
            fn=chat_with_status,
            inputs=[msg_input, chatbot, doc_filter, auto_route, num_chunks],
            outputs=[chatbot, status_bar]
        ).then(
            lambda: "",
            outputs=[msg_input]
        )

        send_btn.click(
            fn=chat_with_status,
            inputs=[msg_input, chatbot, doc_filter, auto_route, num_chunks],
            outputs=[chatbot, status_bar]
        ).then(
            lambda: "",
            outputs=[msg_input]
        )

        clear_chat_btn.click(
            lambda: [],
            outputs=[chatbot]
        )

        example_btn1.click(
            fn=ask_summary,
            inputs=[chatbot],
            outputs=[chatbot]
        ).then(
            fn=update_status_bar,
            outputs=[status_bar]
        )

        example_btn2.click(
            fn=ask_lot_numbers,
            inputs=[chatbot],
            outputs=[chatbot]
        ).then(
            fn=update_status_bar,
            outputs=[status_bar]
        )

        # Auto-process when PDF is uploaded
        pdf_input.change(
            fn=process_pdf_with_status,
            inputs=[pdf_input],
            outputs=[status_output, structure_output, doc_filter, status_bar]
        )

    return demo

In [ ]:
# ============================================
# STEP 11: Launch the Application
# ============================================
#
# WHAT WE'RE DOING:
# Creating the Gradio interface instance and launching it. The share=True
# flag generates a public temporary URL so anyone with the link can access
# the running app from outside Colab. debug=True prints server-side logs
# to the cell output so you can see what happens when the PDF is processed
# and when questions are asked.
#
# WHY THIS MATTERS:
# This is the entry point that makes everything visible and interactive.
# Without this cell, all the pipeline code above would be defined but
# nothing would be rendered for the user. Once launched, the app stays
# alive until you stop the cell or the Colab session times out.
#
# WHAT YOU'LL SEE:
# A public Gradio URL (https://....gradio.live) and an inline iframe
# showing the Pharmaceutical Document Q&A System. Open the URL or use the
# iframe, upload pharma-blob-sample.pdf, and start asking questions.
# ============================================

demo = create_interface()
demo.launch(share=True, debug=True, theme=gr.themes.Soft())

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://66cf6b97cff83a61d2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Starting PDF extraction and analysis...
Extracted 10 pages
Analyzing document structure...



llama_print_timings:        load time =    2231.03 ms
llama_print_timings:      sample time =       0.21 ms /     5 runs   (    0.04 ms per token, 23255.81 tokens per second)
llama_print_timings: prompt eval time =    3351.60 ms /   648 tokens (    5.17 ms per token,   193.34 tokens per second)
llama_print_timings:        eval time =    3044.24 ms /     4 runs   (  761.06 ms per token,     1.31 tokens per second)
llama_print_timings:       total time =    6402.03 ms /   652 tokens
Llama.generate: 7 prefix-match hit, remaining 478 prompt tokens to eval


  Page 0: New document detected - Cover Letter



llama_print_timings:        load time =    2231.03 ms
llama_print_timings:      sample time =       0.97 ms /    20 runs   (    0.05 ms per token, 20554.98 tokens per second)
llama_print_timings: prompt eval time =    1619.89 ms /   478 tokens (    3.39 ms per token,   295.08 tokens per second)
llama_print_timings:        eval time =   12303.45 ms /    19 runs   (  647.55 ms per token,     1.54 tokens per second)
llama_print_timings:       total time =   13940.82 ms /   497 tokens
Llama.generate: 7 prefix-match hit, remaining 532 prompt tokens to eval

llama_print_timings:        load time =    2231.03 ms
llama_print_timings:      sample time =       0.17 ms /     4 runs   (    0.04 ms per token, 23952.10 tokens per second)
llama_print_timings: prompt eval time =    9156.65 ms /   532 tokens (   17.21 ms per token,    58.10 tokens per second)
llama_print_timings:        eval time =    2294.19 ms /     3 runs   (  764.73 ms per token,     1.31 tokens per second)
llama_print_timings:   

...

Query routed to: Chain Of Custody (confidence: 0.95)

===== RETRIEVED CHUNKS SENT TO MISTRAL =====

Chunk 1
Document Type: Chain Of Custody
Pages: 9-9
Relevance Score: 0.1462
Text Preview:
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Global Chain of Custody
List of Assemblies Manufactured at Cytiva Eysins Facility in Switzerland
Page 1 of 1
Cytiva Item #
Description
28930182
Low Flow Kit, AKTA ready
28930183
High Flow Kit, AKTA ready
29021085
AKTA ready Gradient Low Flow Section
29021086
AKTA ready Gradient High Flow Section
29477427
High Flow Kit F, Modified, AKTA ready
29184612
High Flow Gradient C, Modified, AKTA ready
All listed assemblies follow the same chain of custody:
1. Raw materials received at Eysins facility (incoming QC)
2. Sub-assembly and final assembly (in-process checks)
3. Final quality release (per product-specific release criteria)
4. Shipped to Cytiva Distribution Center, Marlborough, MA
5. Distributed to end customer upon order
Traceability: Each as


llama_print_timings:        load time =    2231.03 ms
llama_print_timings:      sample time =       8.89 ms /   171 runs   (    0.05 ms per token, 19232.93 tokens per second)
llama_print_timings: prompt eval time =    1538.52 ms /   480 tokens (    3.21 ms per token,   311.99 tokens per second)
llama_print_timings:        eval time =  118167.11 ms /   170 runs   (  695.10 ms per token,     1.44 tokens per second)
llama_print_timings:       total time =  119869.55 ms /   650 tokens
Llama.generate: 7 prefix-match hit, remaining 225 prompt tokens to eval

llama_print_timings:        load time =    2231.03 ms
llama_print_timings:      sample time =       4.83 ms /    96 runs   (    0.05 ms per token, 19871.66 tokens per second)
llama_print_timings: prompt eval time =    1351.45 ms /   225 tokens (    6.01 ms per token,   166.49 tokens per second)
llama_print_timings:        eval time =   64421.47 ms /    95 runs   (  678.12 ms per token,     1.47 tokens per second)
llama_print_timings:   

Query routed to: Change Log (confidence: 0.95)
Using global search across all document types

===== RETRIEVED CHUNKS SENT TO MISTRAL =====

Chunk 1
Document Type: Packaging Specification
Pages: 3-4
Relevance Score: 0.1280
Text Preview:
Packaging Component Specification
Document Number:
PKG-SPEC-2023-0847
Revision:
C
Effective Date:
2023-11-08
Component:
AKTA ready Flow Kit Packaging Assembly
Drawing Reference:
DWG-29477427-PKG-R3
1. Scope
This specification defines the packaging configuration for AKTA ready Flow Kit assemblies, including primary and
secondary packaging components.
2. Materials
Component
Material
Specification
Blister Tray
PETG
ASTM D1003, min 92% clarity
Lid Film
Tyvek 1073B
Per DuPont specification
Secondary Carton
Corrugated
ECT-32, C-flute
Inner Cushion
PE Foam
2 lb density, 1 inch thick
3. Part Numbers
Assembly Part Number:
PKG-29477427-C
Blister Tray:
PKG-TRAY-4477-B
Lid Film:
PKG-LID-1073-A
Secondary Carton:
PKG-BOX-9301-A


Packaging Component Specification (con


llama_print_timings:        load time =    2231.03 ms
llama_print_timings:      sample time =       8.69 ms /   164 runs   (    0.05 ms per token, 18863.58 tokens per second)
llama_print_timings: prompt eval time =    4548.43 ms /  1404 tokens (    3.24 ms per token,   308.68 tokens per second)
llama_print_timings:        eval time =  121214.04 ms /   163 runs   (  743.64 ms per token,     1.34 tokens per second)
llama_print_timings:       total time =  125946.52 ms /  1567 tokens
Llama.generate: 7 prefix-match hit, remaining 232 prompt tokens to eval

llama_print_timings:        load time =    2231.03 ms
llama_print_timings:      sample time =       2.26 ms /    47 runs   (    0.05 ms per token, 20759.72 tokens per second)
llama_print_timings: prompt eval time =    1375.00 ms /   232 tokens (    5.93 ms per token,   168.73 tokens per second)
llama_print_timings:        eval time =   29514.36 ms /    46 runs   (  641.62 ms per token,     1.56 tokens per second)
llama_print_timings:   

Query routed to: Other (confidence: 0.95)
Using global search across all document types

===== RETRIEVED CHUNKS SENT TO MISTRAL =====

Chunk 1
Document Type: Packaging Specification
Pages: 3-4
Relevance Score: 0.1549
Text Preview:
Packaging Component Specification
Document Number:
PKG-SPEC-2023-0847
Revision:
C
Effective Date:
2023-11-08
Component:
AKTA ready Flow Kit Packaging Assembly
Drawing Reference:
DWG-29477427-PKG-R3
1. Scope
This specification defines the packaging configuration for AKTA ready Flow Kit assemblies, including primary and
secondary packaging components.
2. Materials
Component
Material
Specification
Blister Tray
PETG
ASTM D1003, min 92% clarity
Lid Film
Tyvek 1073B
Per DuPont specification
Secondary Carton
Corrugated
ECT-32, C-flute
Inner Cushion
PE Foam
2 lb density, 1 inch thick
3. Part Numbers
Assembly Part Number:
PKG-29477427-C
Blister Tray:
PKG-TRAY-4477-B
Lid Film:
PKG-LID-1073-A
Secondary Carton:
PKG-BOX-9301-A


Packaging Component Specification (continue